# Stage 4 — Dual-stream fusion (landmark + DINOv2 fingertip pool), 5-fold subject CV

Two fusion designs (plan §2 Task C):
1. **Early fusion** — concat `[landmark_190, dino_1153]` per frame → single 4-layer Conformer at d=256.
2. **Late fusion**  — two parallel 3-layer Conformers (d=128 each) over the two streams, concat at the end.

Run order: 5 early-fusion folds, then 5 late-fusion folds (10 runs total).

**Prerequisites**:
- Stage 1 landmark cache  → `skeleton_features_t32.pt`
- Stage 3 DINOv2 cache    → `stage3_features.pt`
- Stage 1 v3 CV manifest  → `subject_cv5.json`

Both caches must be index-aligned (same `samples` stream + same seed); the trainer asserts this on dataset construction.

**Pass/fail** (plan §2 Task C): mean val CER ≤ min(Stage 1 v3, Stage 3) − 0.03 (i.e. ≤ 0.61 against current measured baselines).  Stretch: ≤ 0.55.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy transformers --quiet

import sys, os
BASE = '/content'
WITA_ROOT = os.path.join(BASE, 'wita_v2')
!rm -rf {WITA_ROOT}
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" {WITA_ROOT}
sys.path.insert(0, BASE)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU            : {name}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

## Cell 2 — Drive mount + config (locked Stage 1 v2 hyperparams)

`USE_DRIVE = True` reuses Stage 1 / Stage 3 caches stored in Drive so we don't rebuild them.

In [ ]:
import os, logging, random, json
import numpy as np
import torch

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive'
    STAGE1_CACHE = os.path.join(DRIVE_ROOT, 'wita_v2_stage1v3', 'skeleton_features_t32.pt')
    STAGE3_CACHE = os.path.join(DRIVE_ROOT, 'wita_v2_stage3',   'stage3_features.pt')
    CV_MANIFEST  = os.path.join(DRIVE_ROOT, 'wita_v2_stage1v3', 'subject_cv5.json')
    PERSIST_ROOT = os.path.join(DRIVE_ROOT, 'wita_v2_stage4')
else:
    STAGE1_CACHE = '/content/skeleton_features_t32.pt'
    STAGE3_CACHE = '/content/stage3_features.pt'
    CV_MANIFEST  = '/content/subject_cv5.json'
    PERSIST_ROOT = '/content/wita_v2_stage4'
os.makedirs(PERSIST_ROOT, exist_ok=True)
RESULTS_PATH = os.path.join(PERSIST_ROOT, 'stage4_results.json')

EPHEM = '/content'
os.makedirs(os.path.join(EPHEM, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(EPHEM, 'logs'),        exist_ok=True)

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(EPHEM, 'logs', 'stage4.log'))])

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# Locked from Stage 1 v2 / 1 v3 / 3.
NUM_EPOCHS    = 80
BATCH_SIZE    = 32
LR_PEAK       = 5e-4
WEIGHT_DECAY  = 5e-2
GRAD_CLIP     = 1.0
DROPOUT       = 0.2
WARMUP_PCT    = 0.05
UPSAMPLE      = 2
CONV_KERNEL   = 15
N_HEADS       = 4
SEED          = 42
FOLDS         = list(range(5))
FUSION_MODES  = ['early', 'late']

# Mode-specific shape (overridden per run).
EARLY_D_MODEL   = 256
EARLY_N_LAYERS  = 4
LATE_D_PER      = 128
LATE_LAYERS_PER = 3

cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA', lang='english',
        max_zips=None, max_frames=64,
        train_split=0.90, seed=SEED,
        download_dir=os.path.join(EPHEM, 'hf_downloads'),
    ),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
        num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
        checkpoint_dir=os.path.join(EPHEM, 'checkpoints'),
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device         : {cfg.device}')
print(f'Stage 1 cache  : {STAGE1_CACHE}  (exists={os.path.exists(STAGE1_CACHE)})')
print(f'Stage 3 cache  : {STAGE3_CACHE}  (exists={os.path.exists(STAGE3_CACHE)})')
print(f'CV manifest    : {CV_MANIFEST}  (exists={os.path.exists(CV_MANIFEST)})')
print(f'Results path   : {RESULTS_PATH}')

## Cell 3 — Load both caches + CV manifest (no streaming or extraction here)

In [ ]:
from wita_v2.datasets.cv_splits import load_cv5_manifest
from wita_v2.training.stage4_train import _verify_caches_aligned

assert os.path.exists(STAGE1_CACHE), f'Build the Stage 1 cache first (skeleton_features_t32.pt).'
assert os.path.exists(STAGE3_CACHE), f'Build the Stage 3 cache first (stage3_features.pt).'
assert os.path.exists(CV_MANIFEST),  f'Build the CV manifest first (subject_cv5.json).'

cache_l = torch.load(STAGE1_CACHE, map_location='cpu', weights_only=False)
cache_d = torch.load(STAGE3_CACHE, map_location='cpu', weights_only=False)
_verify_caches_aligned(cache_l, cache_d)
manifest = load_cv5_manifest(CV_MANIFEST)

print(f'Landmark cache : {len(cache_l["feats"])} clips, in_dim={cache_l["out_dim"]}')
print(f'DINOv2 cache   : {len(cache_d["feats"])} clips, in_dim={cache_d["out_dim"]}')
print(f'CV manifest    : {manifest["n_subjects_total"]} signers across {manifest["n_folds"]} folds')

## Cell 4 — Run the 10-run sweep (5 folds × 2 fusion modes)

Loop order: mode-major (all 5 early folds, then all 5 late folds) so a partial completion still gives one fully-aggregable variant.  Resume logic skips completed `(fold, variant)` pairs.

In [ ]:
from wita_v2.training.stage4_train import train_one_fold
from wita_v2.datasets.cv_splits     import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} runs already complete.')
else:
    all_results = []
    completed = set()

# Per-stream augmenters per the configs.
aug_landmark = LandmarkAugment(p_spatial_jitter=0.80, spatial_sigma=0.02)
aug_dino     = LandmarkAugment(p_spatial_jitter=0.0,  spatial_sigma=0.0)

subject_per_idx = cache_l['subjects']
for fusion_mode in FUSION_MODES:
    variant_name = f'stage4_{fusion_mode}'
    for fold in FOLDS:
        if (fold, variant_name) in completed:
            continue
        train_idx, val_idx = fold_indices(manifest, fold, subject_per_idx)
        kwargs = dict(
            cache_landmark=cache_l, cache_dino=cache_d,
            train_idx=train_idx, val_idx=val_idx, cfg=cfg,
            fold=fold, fusion_mode=fusion_mode, variant=variant_name,
            num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
            weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
            n_heads=N_HEADS, conv_kernel=CONV_KERNEL, upsample=UPSAMPLE,
            warmup_pct=WARMUP_PCT,
            transform_landmark=aug_landmark, transform_dino=aug_dino,
        )
        if fusion_mode == 'early':
            kwargs.update(d_model=EARLY_D_MODEL, n_layers=EARLY_N_LAYERS)
        else:
            kwargs.update(d_per_stream=LATE_D_PER, n_layers_per_stream=LATE_LAYERS_PER)
        result = train_one_fold(**kwargs)
        summary = {k: v for k, v in result.items() if k != 'history'}
        all_results.append(summary)
        with open(RESULTS_PATH, 'w') as f:
            json.dump(all_results, f, indent=2)
        print(f'  saved → {RESULTS_PATH}  (total runs: {len(all_results)})\n')
print(f'\nAll {len(all_results)}/{len(FUSION_MODES)*len(FOLDS)} runs done.')

## Cell 5 — Aggregate: per-fold table, mean ± std, paired Wilcoxon vs Stage 3

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

with open(RESULTS_PATH) as f:
    all_results = json.load(f)

by_variant = {f'stage4_{m}': [None]*len(FOLDS) for m in FUSION_MODES}
for r in all_results:
    if r['variant'] in by_variant:
        by_variant[r['variant']][r['fold']] = r['best_val_cer']

print('Per-fold best val CER\n')
print(' fold   stage4_early   stage4_late')
for fold in FOLDS:
    e = by_variant['stage4_early'][fold]
    l = by_variant['stage4_late'][fold]
    e_s = f'{e:>10.4f}' if e is not None else f'{"-":>10}'
    l_s = f'{l:>10.4f}' if l is not None else f'{"-":>10}'
    print(f'  {fold:>2d}    {e_s}    {l_s}')

stats = {}
for vname, vals in by_variant.items():
    arr = np.array([v for v in vals if v is not None])
    if len(arr):
        stats[vname] = (arr.mean(), arr.std(ddof=0), len(arr))
        print(f'\n{vname}: {arr.mean():.4f} ± {arr.std(ddof=0):.4f}  (n={len(arr)})')

## Cell 6 — Verdict per plan §2 Task C

In [ ]:
import numpy as np
STAGE1V3_NO_DANN = 0.6448    # measured 5-fold mean
STAGE3_MEAN      = None      # filled in once you run the Stage 3 notebook

if STAGE3_MEAN is None:
    print('Set STAGE3_MEAN in this cell from the Stage 3 verdict before running.')
else:
    threshold = min(STAGE1V3_NO_DANN, STAGE3_MEAN) - 0.03
    print(f'\n=== Stage 4 verdict ===')
    print(f'  Stage 1 v3 baseline : {STAGE1V3_NO_DANN:.4f}')
    print(f'  Stage 3 baseline    : {STAGE3_MEAN:.4f}')
    print(f'  Fusion threshold    : ≤ {threshold:.4f}\n')
    for vname, (m, s, n) in stats.items():
        if m <= threshold - (0.55 - threshold):
            tag = '✅ STRETCH ≤ 0.55'
        elif m <= threshold:
            tag = '✅ HEADLINE'
        else:
            tag = '❌ does not beat threshold'
        print(f'  {vname:<15s}: {m:.4f} ± {s:.4f}  {tag}')
    e = [v for v in by_variant['stage4_early'] if v is not None]
    l = [v for v in by_variant['stage4_late']  if v is not None]
    if len(e) == len(l) >= 2:
        paired = [(a, b) for a, b in zip(by_variant['stage4_early'], by_variant['stage4_late'])
                  if a is not None and b is not None]
        if len(paired) >= 2:
            a = np.array([p[0] for p in paired]); b = np.array([p[1] for p in paired])
            try:
                W, p = wilcoxon(a, b, zero_method='zsplit', alternative='two-sided')
                print(f'\n  Paired Wilcoxon early vs late  W={W:.2f}  p={p:.4f}  n={len(paired)}')
                print(f'  diff (early - late) mean = {float((a-b).mean()):+.4f}')
            except ValueError as e:
                print(f'  Wilcoxon failed: {e}')

## Cell 7 — Per-signer val CER scatter (fusion vs baseline)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer = {f'stage4_{m}': {} for m in FUSION_MODES}
for r in all_results:
    if r['variant'] in per_signer:
        per_signer[r['variant']].update(r.get('best_per_signer_val_cer', {}) or {})
all_signers = sorted(set().union(*[d.keys() for d in per_signer.values()]))
HARD = ['PHW','PJH','SYB','KJM','KNY','LKS','KIM','YMG']
colors = {'stage4_early': '#2ca02c', 'stage4_late': '#9467bd'}
fig, ax = plt.subplots(figsize=(11, 4.5))
for vname, d in per_signer.items():
    xs = list(range(len(all_signers)))
    ys = [d.get(s, float('nan')) for s in all_signers]
    ax.scatter(xs, ys, label=vname, s=30, alpha=0.8, color=colors.get(vname))
ax.axhline(0.55, color='green', linestyle='--', alpha=0.5, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.5, label='hard ≥ 0.75')
ax.set_xticks(range(len(all_signers)))
ax.set_xticklabels(all_signers, rotation=90, fontsize=7)
ax.set_ylabel('Val CER at best-epoch')
ax.set_title('Stage 4 fusion — per-signer val CER across folds')
ax.legend(frameon=False)
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EPHEM, 'logs', 'stage4_per_signer_scatter.png'), dpi=140)
plt.show()